In [1]:
!pip install -q langchain langchain-core langchain-community pydantic duckduckgo-search ddgs langchain-experimental requests langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 91.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [68]:
from google.colab import userdata
import os

hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [69]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import requests
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [70]:
# tool create

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url=f'https://v6.exchangerate-api.com/v6/657443b10aec0f33bcd4fd5c/pair/{base_currency}/{target_currency}'
  response=requests.get(url)
  return response.json()

In [71]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1781222401,
 'time_last_update_utc': 'Fri, 12 Jun 2026 00:00:01 +0000',
 'time_next_update_unix': 1781308801,
 'time_next_update_utc': 'Sat, 13 Jun 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6773}

In [72]:
# tool create

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """ Given a currency  conversion rate this function calculates the target currency value from a given base currency value"""
  return base_currency_value*conversion_rate

In [73]:
convert.invoke({'base_currency_value':80,'conversion_rate':95})

7600

In [74]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)

llm = ChatHuggingFace(llm=llm)

In [75]:
# tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [76]:
currency_system_prompt = """
You are a currency conversion assistant.

Rules:
1. If the user asks "convert X A to B", then:
   - base_currency = A
   - target_currency = B
   - amount = X

2. If the user asks only for "conversion factor between A and B", then:
   - base_currency = A
   - target_currency = B

3. If the user asks both conversion factor and amount conversion, use the amount conversion direction as the main direction.
   Example:
   "conversion factor between INR and USD and convert 10 USD to INR"
   means:
   - base_currency = USD
   - target_currency = INR
   - amount = 10

4. Do not reverse the direction unless the user clearly says so.
"""

In [77]:
messages = [
    SystemMessage(content=currency_system_prompt),
    HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR')
    ]

In [78]:
messages

[SystemMessage(content='\nYou are a currency conversion assistant.\n\nRules:\n1. If the user asks "convert X A to B", then:\n   - base_currency = A\n   - target_currency = B\n   - amount = X\n\n2. If the user asks only for "conversion factor between A and B", then:\n   - base_currency = A\n   - target_currency = B\n\n3. If the user asks both conversion factor and amount conversion, use the amount conversion direction as the main direction.\n   Example:\n   "conversion factor between INR and USD and convert 10 USD to INR"\n   means:\n   - base_currency = USD\n   - target_currency = INR\n   - amount = 10\n\n4. Do not reverse the direction unless the user clearly says so.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [79]:
ai_msg=llm_with_tools.invoke(messages)

In [80]:
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_8f259m9vvyp2hj8t2oa5uo2q', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert', 'description': None}, 'id': 'call_hajy3dzzmylj1jdfmi6yja7g', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 490, 'total_tokens': 542}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ebad4-2465-7443-a887-71f933e203e8-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_8f259m9vvyp2hj8t2oa5uo2q', 'type': 'tool_call'}, {'name': 'convert', 'args': {'base_currency_value': 10}, 'id': 'call_hajy3dzzmylj1jdfmi6yja7g', 'type': 'tool_call'}], invalid_tool_calls=[], usage

In [81]:
ai_msg.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_8f259m9vvyp2hj8t2oa5uo2q',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_hajy3dzzmylj1jdfmi6yja7g',
  'type': 'tool_call'}]

In [82]:
messages.append(ai_msg)

In [83]:
import json

for tool_call in ai_msg.tool_calls:

  #execute the 1st tool and get converison rate
  if tool_call['name']=='get_conversion_factor':
    tool_msg1=get_conversion_factor.invoke(tool_call)

    # convert to json and fetch conversion rate
    conversion_rate=json.loads(tool_msg1.content)['conversion_rate']

    messages.append(tool_msg1)

  #execute 2nd tool
  if tool_call['name']=='convert':
    #fetch curr arg
    tool_call['args']['conversion_rate']=conversion_rate
    tool_msg2=convert.invoke(tool_call)
    messages.append(tool_msg2)


In [84]:
messages

[SystemMessage(content='\nYou are a currency conversion assistant.\n\nRules:\n1. If the user asks "convert X A to B", then:\n   - base_currency = A\n   - target_currency = B\n   - amount = X\n\n2. If the user asks only for "conversion factor between A and B", then:\n   - base_currency = A\n   - target_currency = B\n\n3. If the user asks both conversion factor and amount conversion, use the amount conversion direction as the main direction.\n   Example:\n   "conversion factor between INR and USD and convert 10 USD to INR"\n   means:\n   - base_currency = USD\n   - target_currency = INR\n   - amount = 10\n\n4. Do not reverse the direction unless the user clearly says so.\n', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"U

In [86]:
llm_with_tools.invoke(messages).content

'The conversion factor from USD to INR is approximately 95.6773. Based on this, 10 USD is equivalent to approximately 956.773 INR.'